In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
from forestplot import forestplot
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
plt.rcParams['font.monospace'] = ['Noto Mono', 'Tlwg Mono', 'Courier New']


from helpers import save_mpl_fig

In [ ]:
df = pd.read_parquet("gc_human_tweets.parquet").assign(
    mentions=lambda df_: df_["mentions"].apply(
        lambda x: x.tolist() if isinstance(x, np.ndarray) else x
    )
)
df

In [ ]:
# Count all mentions
all_mentions = Counter()
for mentions in df["mentions"]:
    all_mentions.update(mentions)

df_at = (
    pd.DataFrame.from_dict(Counter(all_mentions), orient="index", columns=["count"])
    .sort_values("count", ascending=False)
    .reset_index(names="at")
)
df_at.to_csv("output/at.csv", index=False)
df_at.head(20)

In [ ]:
TOP_N = 30
top_at = Counter(all_mentions).most_common(TOP_N)
at_metadata = []

for at, count in top_at:
    # Get all tweets containing this at
    tweets_with_at = df[
        df["mentions"].apply(lambda x: at in x if isinstance(x, list) else False)
    ]

    # Calculate metrics
    pct = (count / len(df)) * 100
    mean_retweets = tweets_with_at["retweets"].mean()
    mean_favs = tweets_with_at["favorites"].mean()

    # Calculate mean number of ats per tweet
    at_counts = tweets_with_at["mentions"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    mean_ats = at_counts.mean()

    at_metadata.append(
        {
            "at": at,
            "count": count,
            "pct": pct,
            "mean_ats": mean_ats,
            "mean_retweets": mean_retweets,
            "mean_favs": mean_favs,
        }
    )

df_plot = pd.DataFrame(at_metadata)

# Create formatted strings with proper alignment
df_plot["count_str"] = df_plot["count"].apply(lambda x: f"{x:,}".rjust(6))
df_plot["pct_str"] = df_plot["pct"].apply(lambda x: f"{x:.2f}%".rjust(8))
df_plot["rt_str"] = df_plot["mean_retweets"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["fav_str"] = df_plot["mean_favs"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["ht_str"] = df_plot["mean_ats"].apply(lambda x: f"{x:.1f}".rjust(5))

# For forestplot - use pct as both estimate and limits (creates dot plot)
df_plot["ll"] = df_plot["pct"]
df_plot["hl"] = df_plot["pct"]

df_plot[["at", "count", "pct", "mean_retweets", "mean_favs", "mean_ats"]].head(10)

df_plot

In [ ]:
ax = forestplot(
    df_plot,
    estimate="pct",  # What to plot (percentage)
    ll="ll",  # Lower limit (same as estimate for dot plot)
    hl="hl",  # Upper limit (same as estimate for dot plot)
    varlabel="at",  # Y-axis labels (hashtag names)
    annote=["rt_str", "fav_str"],  # Left-side annotations
    annoteheaders=[" RT", "Fave"],  # Left annotation headers
    rightannote=["count_str", "pct_str"],  # Right-side annotations
    right_annoteheaders=["Tweets", "Tweets(%)"],  # Right annotation headers
    ci_report=False,  # No confidence intervals
    figsize=(5, TOP_N * 0.4),  # Scale height with number of hashtags
    color_alt_rows=True,  # Alternate row colors for readability
    table=True,  # Show as table
    xticks=[0, 0.2, 0.4, 0.6, 0.8, 1],  # X-axis ticks (adjust based on your max %)
    **{
        "marker": "o",
        "variable_header": "Mention",
        "markersize": 40,
        "xlinecolor": ".8",
        # Font size controls
        "xtick_size": 11,  # X-axis tick numbers
    },
)

# Format x-axis as percentages
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{x:.1f}%"))
ax.set_xlabel("Percentage of #MeToo tweets", fontsize=12)

# Add subtle gridlines at major percentages
for vline in [0, 0.2, 0.4, 0.6, 0.8, 1]:
    ax.vlines(vline, ymin=-0.5, ymax=TOP_N - 0.5, colors=".85", linewidth=0.5)

save_mpl_fig("output/figures/top_mentions_tabulation")